# 07 — DDPM From Scratch

> **Notebook 06:** We learned what diffusion is.  
> **Notebook 07:** We build it.

The goal is to turn the diffusion equations into a working model that can learn to predict the noise hidden inside a noisy image.

```text
Forward process:

x₀ → x₁ → x₂ → ... → x_T
```

```text
Reverse process:

x_T → x_{T-1} → ... → x₁ → x₀
```

The practical question is:

> **How do we learn the reverse process?**

We train a neural network to predict the Gaussian noise that was added to a clean image:

```text
x₀ + ε + t
      ↓
     x_t
      ↓
 U-Net ε_θ(x_t, t)
      ↓
 predicted noise
      ↓
 compare with true ε
```

This notebook intentionally focuses on implementation. The mathematics is recalled only when a formula is needed by the code.

## 2 — Imports, Reproducibility, and Device

Before touching the model, we make the notebook deterministic where practical and choose CPU/CUDA automatically.

In [ ]:
import os
import math
import random
from copy import deepcopy

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Device: {DEVICE}")

## 3 — Dataset Path and DDPM Configuration

The notebook uses the existing local STL-10 dataset. Nothing is downloaded automatically.

For this educational run we use a compact model and `T=300` diffusion steps. The goal is to make the mechanics visible rather than chase state-of-the-art sample quality.

In [ ]:
DATA_ROOT = r"E:\Generative_AI\Educational\data\STL-10"

IMAGE_SIZE = 64
CHANNELS = 3
BATCH_SIZE = 32

T = 300
BETA_START = 1e-4
BETA_END = 0.02

LATENT_BASE_CHANNELS = 64
TIME_EMBED_DIM = 256
LEARNING_RATE = 2e-4
EPOCHS = 20

# Set this to False when you want to explore every section without a full training run.
RUN_TRAINING = True

NUM_WORKERS = 0

print(f"Dataset path: {DATA_ROOT}")
print(f"Training enabled: {RUN_TRAINING}")

## 4 — STL-10 Dataset

STL-10 provides 96×96 RGB images. We resize them to 64×64 so the educational U-Net remains practical on a consumer GPU.

If the path is wrong, fail loudly rather than silently switching to another dataset.

In [ ]:
if not os.path.exists(DATA_ROOT):
    raise FileNotFoundError(
        "STL-10 was not found at the expected path:\n"
        f"{DATA_ROOT}\n\n"
        "Place the extracted STL-10 dataset there or change DATA_ROOT "
        "in this notebook. The notebook will not download a replacement dataset automatically."
    )

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

dataset = datasets.STL10(
    root=DATA_ROOT,
    split="train",
    transform=transform,
    download=False,
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    drop_last=True,
)

print(f"Number of training images: {len(dataset)}")
print(f"Number of batches / epoch: {len(loader)}")

## 5 — A Small Visualization Helper

Training tensors live in `[-1, 1]`. For display we map them back to `[0, 1]`.

The helper intentionally does **not** depend on `torchvision.utils.make_grid`, so the plotting logic stays visible.

In [ ]:
def _to_display_tensor(images):
    """Convert image tensor(s) from [-1, 1] to [0, 1] on CPU."""
    if images.ndim == 3:
        images = images.unsqueeze(0)
    if images.ndim != 4 or images.shape[1] != 3:
        raise ValueError(f"Expected [B, 3, H, W] or [3, H, W], got {tuple(images.shape)}")
    images = images.detach().cpu().float()
    return ((images + 1.0) / 2.0).clamp(0.0, 1.0)


def show_images(images, title=None, rows=4, cols=4, figsize=(8, 8)):
    """Show a clean image grid for RGB tensors in [-1, 1]."""
    images = _to_display_tensor(images)
    total = min(images.shape[0], rows * cols)

    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.atleast_1d(axes).ravel()

    for i, ax in enumerate(axes):
        ax.axis("off")
        if i < total:
            image = images[i].permute(1, 2, 0).numpy()
            ax.imshow(image)

    if title:
        fig.suptitle(title, fontsize=14, y=0.98)
    plt.tight_layout()
    plt.show()
    return fig, axes

In [ ]:
real_batch, real_labels = next(iter(loader))
print("Batch shape:", tuple(real_batch.shape))
print("Label shape:", tuple(real_labels.shape))
assert real_batch.shape[1:] == (CHANNELS, IMAGE_SIZE, IMAGE_SIZE)

show_images(
    real_batch[:16],
    title="Real STL-10 Images (before training)",
    rows=4,
    cols=4,
    figsize=(8, 8),
)

### Why `[-1, 1]`?

A symmetric range makes the image representation centered around zero, which is convenient because the diffusion process itself adds zero-mean Gaussian noise. The important part is consistency: **training, denoising, and visualization must agree on the same normalization convention.**

```text
96 × 96 × 3  →  64 × 64 × 3  →  [-1, 1]

PyTorch tensor:
[B, 3, 64, 64]
```

## 6 — DDPM Noise Schedule

We use the notation from Notebook 06 but with one implementation convention:

> **Python timesteps are indexed `0 ... T-1`.**

Conceptual DDPM papers often write `t = 1 ... T`. Here Python index `0` corresponds to the first reverse/forward step in our arrays. Keeping this convention consistent prevents off-by-one errors.

We precompute all schedule terms because the same quantities are used repeatedly during training and sampling.

In [ ]:
# Python indexing convention: t in {0, ..., T-1}.
# Conceptually this corresponds to diffusion steps 1, ..., T.

betas = torch.linspace(BETA_START, BETA_END, T, dtype=torch.float32, device=DEVICE)
alphas = 1.0 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

sqrt_alpha_bars = torch.sqrt(alpha_bars)
sqrt_one_minus_alpha_bars = torch.sqrt(1.0 - alpha_bars)
sqrt_recip_alphas = torch.sqrt(1.0 / alphas)

alpha_bars_prev = torch.cat([
    torch.ones(1, device=DEVICE, dtype=alpha_bars.dtype),
    alpha_bars[:-1],
])

posterior_variance = betas * (1.0 - alpha_bars_prev) / (1.0 - alpha_bars)
posterior_mean_coef1 = (
    betas * torch.sqrt(alpha_bars_prev) / (1.0 - alpha_bars)
)
posterior_mean_coef2 = (
    (1.0 - alpha_bars_prev) * torch.sqrt(alphas) / (1.0 - alpha_bars)
)

schedule = {
    "betas": betas,
    "alphas": alphas,
    "alpha_bars": alpha_bars,
    "sqrt_alpha_bars": sqrt_alpha_bars,
    "sqrt_one_minus_alpha_bars": sqrt_one_minus_alpha_bars,
    "sqrt_recip_alphas": sqrt_recip_alphas,
    "posterior_variance": posterior_variance,
    "posterior_mean_coef1": posterior_mean_coef1,
    "posterior_mean_coef2": posterior_mean_coef2,
}

print("Schedule tensors:")
for name, tensor in schedule.items():
    print(f"  {name:28s} {tuple(tensor.shape)}")

In [ ]:
t = torch.arange(T, device=DEVICE).detach().cpu().numpy()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t, betas.detach().cpu().numpy(), label="β_t")
ax.plot(t, alphas.detach().cpu().numpy(), label="α_t")
ax.plot(t, alpha_bars.detach().cpu().numpy(), label="ᾱ_t")
ax.set_title("Noise Schedule: β_t, α_t, ᾱ_t")
ax.set_xlabel("Python timestep")
ax.set_ylabel("Value")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t, sqrt_alpha_bars.detach().cpu().numpy(), label="sqrt(ᾱ_t)")
ax.plot(t, sqrt_one_minus_alpha_bars.detach().cpu().numpy(), label="sqrt(1-ᾱ_t)")
ax.set_title("Signal vs Noise Coefficients")
ax.set_xlabel("Python timestep")
ax.set_ylabel("Coefficient")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 7 — `extract`: Turning `[T]` Into `[B, 1, 1, 1]`

At each batch element, the timestep is different. We therefore need to select one scalar from a schedule tensor for every sample and reshape it so it broadcasts over image channels and spatial dimensions.

```text
values     : [T]
timesteps  : [B]

extract(...) → [B, 1, 1, 1]
```

In [ ]:
def extract(values, timesteps, x_shape):
    """Gather one schedule value per batch element and reshape for broadcasting."""
    if timesteps.ndim != 1:
        raise ValueError(f"timesteps must have shape [B], got {tuple(timesteps.shape)}")
    if values.ndim != 1 or values.shape[0] != T:
        raise ValueError(f"values must have shape [T], got {tuple(values.shape)}")

    out = values.gather(0, timesteps)
    return out.reshape(timesteps.shape[0], *((1,) * (len(x_shape) - 1)))


test_x = torch.randn(4, CHANNELS, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
test_t = torch.tensor([0, 1, T // 2, T - 1], device=DEVICE, dtype=torch.long)
test_coeff = extract(sqrt_alpha_bars, test_t, test_x.shape)

print("x shape:", tuple(test_x.shape))
print("t shape:", tuple(test_t.shape))
print("extracted coefficient shape:", tuple(test_coeff.shape))
assert test_coeff.shape == (4, 1, 1, 1)

## 8 — Forward Sampling: `q_sample`

The closed-form forward process is:

$$
x_t = \sqrt{\bar{\alpha}_t}x_0 + \sqrt{1-\bar{\alpha}_t}\,\epsilon,
\qquad \epsilon \sim \mathcal N(0, I)
$$

This lets us jump directly from a clean image `x₀` to any selected timestep without explicitly constructing every intermediate image.

In [ ]:
def q_sample(x0, t, noise=None):
    """Sample x_t directly from x_0 using the closed-form DDPM equation."""
    if noise is None:
        noise = torch.randn_like(x0)
    if noise.shape != x0.shape:
        raise ValueError("noise must have exactly the same shape as x0")

    sqrt_ab = extract(sqrt_alpha_bars, t, x0.shape)
    sqrt_one_minus_ab = extract(sqrt_one_minus_alpha_bars, t, x0.shape)
    return sqrt_ab * x0 + sqrt_one_minus_ab * noise


x0_demo = real_batch[:4].to(DEVICE)
t_demo = torch.tensor([0, 50, 150, T - 1], device=DEVICE, dtype=torch.long)
noise_demo = torch.randn_like(x0_demo)
xt_demo = q_sample(x0_demo, t_demo, noise=noise_demo)

print("x0:", tuple(x0_demo.shape))
print("t :", tuple(t_demo.shape))
print("ε :", tuple(noise_demo.shape))
print("xt:", tuple(xt_demo.shape))
assert xt_demo.shape == x0_demo.shape

## 9 — Visualization: One Image Through the Forward Process

Here we keep one clean image fixed and use one fixed noise tensor. Only the timestep changes, so the figure isolates the effect of the diffusion schedule.

In [ ]:
source_image = real_batch[0:1].to(DEVICE)
fixed_forward_noise = torch.randn_like(source_image)
forward_timesteps = [0, 10, 25, 50, 100, 150, 200, T - 1]

forward_sequence = []
for step in forward_timesteps:
    step_tensor = torch.tensor([step], device=DEVICE, dtype=torch.long)
    forward_sequence.append(q_sample(source_image, step_tensor, noise=fixed_forward_noise))

forward_sequence = torch.cat(forward_sequence, dim=0)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, image, step in zip(axes.ravel(), forward_sequence, forward_timesteps):
    display = _to_display_tensor(image)[0].permute(1, 2, 0).numpy()
    ax.imshow(display)
    ax.set_title(f"t = {step}")
    ax.axis("off")
fig.suptitle("Forward Diffusion: Clean Image → Gaussian Noise", fontsize=15)
plt.tight_layout()
plt.show()

## 10 — Forward Process Across Multiple Images

The same mathematical process is applied to every image in the training distribution. Rows are different STL-10 examples; columns are different timesteps.

In [ ]:
multi_real = real_batch[:4].to(DEVICE)
multi_noise = torch.randn_like(multi_real)
multi_timesteps = [10, 50, 150, 250, T - 1]

fig, axes = plt.subplots(4, len(multi_timesteps), figsize=(13, 10))
for row in range(4):
    for col, step in enumerate(multi_timesteps):
        step_tensor = torch.full((1,), step, device=DEVICE, dtype=torch.long)
        noisy = q_sample(multi_real[row:row + 1], step_tensor, noise=multi_noise[row:row + 1])
        image = _to_display_tensor(noisy)[0].permute(1, 2, 0).numpy()
        axes[row, col].imshow(image)
        axes[row, col].axis("off")
        if row == 0:
            axes[row, col].set_title(f"t = {step}")

fig.suptitle("Forward Diffusion Across Different Images", fontsize=15)
plt.tight_layout()
plt.show()

## 11 — Time Embeddings

The U-Net must know **which noise level** it is denoising. Two inputs can have identical spatial shape while representing completely different denoising tasks.

We convert a scalar timestep into a vector using sinusoidal features. The resulting embedding has shape:

```text
[B, embedding_dim]
```

The idea is related to positional encodings in Transformers: a scalar position is converted into a richer vector representation. Here, the position is the diffusion timestep.

In [ ]:
def sinusoidal_embedding(timesteps, dim):
    """Return sinusoidal timestep embeddings with shape [B, dim]."""
    if timesteps.ndim != 1:
        raise ValueError(f"timesteps must have shape [B], got {tuple(timesteps.shape)}")

    half = dim // 2
    device = timesteps.device
    dtype = torch.float32

    if half == 0:
        raise ValueError("Embedding dimension must be at least 2")

    exponent = torch.arange(half, device=device, dtype=dtype)
    exponent = -math.log(10000.0) * exponent / max(half - 1, 1)
    frequencies = torch.exp(exponent)

    angles = timesteps.to(dtype).reshape(-1, 1) * frequencies.reshape(1, -1)
    embedding = torch.cat([torch.sin(angles), torch.cos(angles)], dim=1)

    if dim % 2 == 1:
        embedding = F.pad(embedding, (0, 1))

    return embedding


small_t = torch.tensor([0, 25, 150, T - 1], device=DEVICE, dtype=torch.long)
small_emb = sinusoidal_embedding(small_t, TIME_EMBED_DIM)
print("Embedding shape:", tuple(small_emb.shape))
assert small_emb.shape == (len(small_t), TIME_EMBED_DIM)

In [ ]:
embedding_timesteps = torch.tensor([0, 25, 50, 100, 150, 200, 299], device=DEVICE, dtype=torch.long)
embedding_matrix = sinusoidal_embedding(embedding_timesteps, 128).detach().cpu().numpy()

fig, ax = plt.subplots(figsize=(13, 4))
im = ax.imshow(embedding_matrix, aspect="auto", interpolation="nearest")
ax.set_title("Sinusoidal Timestep Embeddings")
ax.set_xlabel("Embedding dimension")
ax.set_ylabel("Selected timestep")
ax.set_yticks(range(len(embedding_timesteps)))
ax.set_yticklabels([str(int(v)) for v in embedding_timesteps.detach().cpu()])
fig.colorbar(im, ax=ax, fraction=0.02, pad=0.02, label="Embedding value")
plt.tight_layout()
plt.show()

## 12 — Why a U-Net?

Denoising needs both **local detail** and **broader context**. A compact U-Net gives the model a sequence of lower-resolution feature maps where larger receptive fields are available, while skip connections carry spatial information forward to the decoder.

```text
input noisy image
      ↓
   encoder
      ↓
lower resolution / richer features
      ↓
  bottleneck
      ↓
   decoder
      ↓
noise prediction at full resolution

skip connections: encoder features → matching decoder stages
```

The model is not asked to output an RGB reconstruction directly. Its target is the **noise tensor** that was used to create `x_t`.

## 13 — Residual Block With Time Conditioning

Each residual block follows the pattern:

```text
x
↓
Conv → GroupNorm → SiLU
↓
add projected time embedding
↓
Conv → GroupNorm
↓
residual / skip connection
↓
output
```

Injecting time information inside the block lets the feature processing adapt to the current noise level at multiple depths of the network.

In [ ]:
def make_group_norm(channels, max_groups=8):
    groups = min(max_groups, channels)
    while channels % groups != 0 and groups > 1:
        groups -= 1
    return nn.GroupNorm(groups, channels)


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim):
        super().__init__()
        self.norm1 = make_group_norm(in_channels)
        self.act1 = nn.SiLU()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)

        self.time_proj = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_dim, out_channels),
        )

        self.norm2 = make_group_norm(out_channels)
        self.act2 = nn.SiLU()
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)

        self.skip = (
            nn.Conv2d(in_channels, out_channels, kernel_size=1)
            if in_channels != out_channels
            else nn.Identity()
        )

    def forward(self, x, time_embedding):
        h = self.conv1(self.act1(self.norm1(x)))
        time_bias = self.time_proj(time_embedding).unsqueeze(-1).unsqueeze(-1)
        h = h + time_bias
        h = self.conv2(self.act2(self.norm2(h)))
        return h + self.skip(x)

## 14 — Compact Educational DDPM U-Net

To reach the requested 8×8 bottleneck from a 64×64 input, we use **three** downsampling stages:

```text
Input      [B,   3, 64, 64]
Down 1     [B,  64, 32, 32]
Down 2     [B, 128, 16, 16]
Down 3     [B, 256,  8,  8]
Bottleneck [B, 256,  8,  8]
Up 3       [B, 128, 16, 16]
Up 2       [B,  64, 32, 32]
Up 1       [B,  64, 64, 64]
Output     [B,   3, 64, 64]
```

The implementation uses Conv2d, GroupNorm and SiLU. No BatchNorm or attention blocks are introduced here.

In [ ]:
class Downsample(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, kernel_size=4, stride=2, padding=1)

    def forward(self, x):
        return self.conv(x)


class Upsample(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.ConvTranspose2d(
            in_channels,
            out_channels,
            kernel_size=4,
            stride=2,
            padding=1,
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, base_channels=64, time_dim=256):
        super().__init__()
        c1 = base_channels
        c2 = base_channels * 2
        c3 = base_channels * 4

        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, time_dim * 4),
            nn.SiLU(),
            nn.Linear(time_dim * 4, time_dim),
        )

        self.input_conv = nn.Conv2d(in_channels, c1, kernel_size=3, padding=1)

        self.down1_block = ResidualBlock(c1, c1, time_dim)
        self.down1 = Downsample(c1)

        self.down2_block = ResidualBlock(c1, c2, time_dim)
        self.down2 = Downsample(c2)

        self.down3_block = ResidualBlock(c2, c3, time_dim)
        self.down3 = Downsample(c3)

        self.mid1 = ResidualBlock(c3, c3, time_dim)
        self.mid2 = ResidualBlock(c3, c3, time_dim)

        self.up3 = Upsample(c3, c2)
        self.up3_block = ResidualBlock(c2 + c2, c2, time_dim)

        self.up2 = Upsample(c2, c1)
        self.up2_block = ResidualBlock(c1 + c1, c1, time_dim)

        self.up1 = Upsample(c1, c1)
        self.up1_block = ResidualBlock(c1 + c1, c1, time_dim)

        self.out_norm = make_group_norm(c1)
        self.out_act = nn.SiLU()
        self.out_conv = nn.Conv2d(c1, in_channels, kernel_size=3, padding=1)

    def forward(self, x, timesteps):
        time_embedding = sinusoidal_embedding(timesteps, self.time_mlp[0].in_features)
        time_embedding = self.time_mlp(time_embedding)

        x = self.input_conv(x)

        skip1 = self.down1_block(x, time_embedding)   # 64×64
        x = self.down1(skip1)                          # 32×32

        skip2 = self.down2_block(x, time_embedding)   # 32×32
        x = self.down2(skip2)                          # 16×16

        skip3 = self.down3_block(x, time_embedding)   # 16×16
        x = self.down3(skip3)                          # 8×8

        x = self.mid1(x, time_embedding)
        x = self.mid2(x, time_embedding)

        x = self.up3(x)                               # 16×16
        x = torch.cat([x, skip3], dim=1)
        x = self.up3_block(x, time_embedding)

        x = self.up2(x)                               # 32×32
        x = torch.cat([x, skip2], dim=1)
        x = self.up2_block(x, time_embedding)

        x = self.up1(x)                               # 64×64
        x = torch.cat([x, skip1], dim=1)
        x = self.up1_block(x, time_embedding)

        return self.out_conv(self.out_act(self.out_norm(x)))


model = UNet(
    in_channels=CHANNELS,
    base_channels=LATENT_BASE_CHANNELS,
    time_dim=TIME_EMBED_DIM,
).to(DEVICE)

## 15 — U-Net Tensor Shape Walkthrough

The important part is not just the architecture diagram; it is knowing what every tensor looks like.

```text
Input                 [B,   3, 64, 64]
Encoder stage 1       [B,  64, 64, 64]
↓ downsample
Encoder stage 2       [B,  64, 32, 32]
↓ downsample
Encoder stage 3       [B, 128, 16, 16]
↓ downsample
Bottleneck             [B, 256,  8,  8]
↑ upsample + skip      [B, 128, 16, 16]
↑ upsample + skip      [B,  64, 32, 32]
↑ upsample + skip      [B,  64, 64, 64]
Output                [B,   3, 64, 64]
```

For example, at the 16×16 stage:

```text
decoder feature  [B, 128, 16, 16]
skip feature     [B, 128, 16, 16]
                     │
                     └── concatenate along channels
                               ↓
                     [B, 256, 16, 16]
```

The residual block then reduces that concatenated representation back to the decoder width.

In [ ]:
def count_parameters(module):
    return sum(parameter.numel() for parameter in module.parameters() if parameter.requires_grad)


x_fake = torch.randn(2, CHANNELS, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
t_fake = torch.randint(0, T, (2,), device=DEVICE, dtype=torch.long)
with torch.no_grad():
    pred_fake = model(x_fake, t_fake)

print("Input shape       :", tuple(x_fake.shape))
print("Timestep shape    :", tuple(t_fake.shape))
print("Output shape      :", tuple(pred_fake.shape))
print("Trainable params  :", f"{count_parameters(model):,}")
assert pred_fake.shape == x_fake.shape

## 16 — U-Net at a Glance

```text
                             time embedding
                                  │
                                  ▼
input → block → ↓ → block → ↓ → block → ↓ → bottleneck
          │                 │                 │
          │                 │                 │
          └──────── skip ───┘──────── skip ───┘

bottleneck → ↑ → concat → block → ↑ → concat → block → ↑ → concat → block → noise
```

The skip connections are what let the decoder combine large-context features with spatial information captured before downsampling.

## 17 — The Noise Prediction Objective

For one training example we have:

```text
x₀ = clean image
t  = sampled timestep
ε  = sampled Gaussian noise
```

We create `x_t`, then ask the U-Net to predict the same noise:

$$
\hat\epsilon = \epsilon_\theta(x_t,t)
$$

The simplified DDPM loss is:

$$
L_{simple} = \left\|\epsilon - \epsilon_\theta(x_t,t)\right\|^2
$$

So the neural network is **not predicting pixels directly**. It learns a noise field that can later be used to estimate the clean image and perform a reverse step.

## 18 — One Complete Training Step

Let's verify the complete data flow before writing a multi-epoch loop.

```text
1. take real image x₀
2. sample random timestep t
3. sample Gaussian noise ε
4. construct x_t
5. feed x_t and t into U-Net
6. predict ε
7. calculate MSE
8. zero gradients
9. backward
10. optimizer step
```

In [ ]:
demo_model = deepcopy(model).to(DEVICE)
demo_optimizer = torch.optim.AdamW(demo_model.parameters(), lr=LEARNING_RATE)

demo_model.train()
demo_optimizer.zero_grad(set_to_none=True)

x0_step = real_batch[:BATCH_SIZE].to(DEVICE, non_blocking=True)
t_step = torch.randint(0, T, (x0_step.shape[0],), device=DEVICE, dtype=torch.long)
noise_step = torch.randn_like(x0_step)
xt_step = q_sample(x0_step, t_step, noise=noise_step)
pred_noise_step = demo_model(xt_step, t_step)
loss_step = F.mse_loss(pred_noise_step, noise_step)
loss_step.backward()
demo_optimizer.step()

del demo_optimizer, demo_model

print("x0 shape            :", tuple(x0_step.shape))
print("t shape             :", tuple(t_step.shape))
print("noise shape         :", tuple(noise_step.shape))
print("x_t shape           :", tuple(xt_step.shape))
print("predicted noise     :", tuple(pred_noise_step.shape))
print("one-step loss       :", float(loss_step.detach().cpu()))
assert pred_noise_step.shape == noise_step.shape

## 19 — Diagnostic: True Noise vs Predicted Noise

These maps are not natural-image reconstructions. They are diagnostics of what the network is predicting in noise space.

We normalize each map independently for display so that the structure of the prediction and its absolute error are easier to see.

In [ ]:
def _normalize_map_for_display(tensor):
    values = tensor.detach().cpu().float()
    lo = values.amin()
    hi = values.amax()
    return ((values - lo) / (hi - lo + 1e-8)).numpy()


sample_idx = 0
true_noise_map = noise_step[sample_idx, 0]
pred_noise_map = pred_noise_step[sample_idx, 0]
error_map = (true_noise_map - pred_noise_map).abs()

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, tensor, title in [
    (axes[0], true_noise_map, "True noise"),
    (axes[1], pred_noise_map, "Predicted noise"),
    (axes[2], error_map, "Absolute error"),
]:
    ax.imshow(_normalize_map_for_display(tensor), cmap="magma")
    ax.set_title(title)
    ax.axis("off")
fig.suptitle("True Noise vs Predicted Noise (diagnostic maps)", fontsize=14)
plt.tight_layout()
plt.show()

## 20 — Recovering a Predicted Clean Image

Once the network predicts noise, we can algebraically rearrange the forward equation to estimate the clean image:

$$
\hat{x}_0 = \frac{x_t - \sqrt{1-\bar\alpha_t}\,\hat\epsilon}{\sqrt{\bar\alpha_t}}
$$

This quantity is useful both for understanding the reverse process and for diagnostics.

In [ ]:
def predict_x0_from_eps(x_t, t, predicted_noise):
    """Recover x0 estimate from x_t and a predicted noise tensor."""
    sqrt_ab = extract(sqrt_alpha_bars, t, x_t.shape)
    sqrt_one_minus_ab = extract(sqrt_one_minus_alpha_bars, t, x_t.shape)
    x0_pred = (x_t - sqrt_one_minus_ab * predicted_noise) / sqrt_ab.clamp_min(1e-8)
    return x0_pred.clamp(-1.0, 1.0)


with torch.no_grad():
    x0_pred_step = predict_x0_from_eps(xt_step, t_step, pred_noise_step)

print("Predicted x0 shape:", tuple(x0_pred_step.shape))
assert x0_pred_step.shape == x0_step.shape

In [ ]:
inspection_timesteps = [10, 75, 150, 225, T - 1]
inspection_index = 0
inspection_x0 = real_batch[inspection_index:inspection_index + 1].to(DEVICE)
inspection_noise = torch.randn_like(inspection_x0)

fig, axes = plt.subplots(len(inspection_timesteps), 3, figsize=(9, 3 * len(inspection_timesteps)))
for row, step in enumerate(inspection_timesteps):
    t_row = torch.tensor([step], device=DEVICE, dtype=torch.long)
    x_t_row = q_sample(inspection_x0, t_row, noise=inspection_noise)
    with torch.no_grad():
        eps_row = model(x_t_row, t_row)
        x0_hat_row = predict_x0_from_eps(x_t_row, t_row, eps_row)

    clean = _to_display_tensor(inspection_x0)[0].permute(1, 2, 0).numpy()
    noisy = _to_display_tensor(x_t_row)[0].permute(1, 2, 0).numpy()
    predicted = _to_display_tensor(x0_hat_row)[0].permute(1, 2, 0).numpy()

    for col, image, title in [
        (0, clean, "original x₀"),
        (1, noisy, f"x_t, t={step}"),
        (2, predicted, "predicted x̂₀"),
    ]:
        axes[row, col].imshow(image)
        axes[row, col].axis("off")
        axes[row, col].set_title(title if row == 0 else "")
    axes[row, 0].set_ylabel(f"t={step}", rotation=0, labelpad=28, va="center")

fig.suptitle("Predicted x₀ at Different Noise Levels", fontsize=15)
plt.tight_layout()
plt.show()

## 21 — Fixed Diagnostic Inputs

For training diagnostics we will reuse the same clean images, timesteps, and noise tensors. That turns the diagnostic into a controlled experiment:

```text
same x₀ + same t + same ε
                ↓
       different model checkpoint
                ↓
       observable model evolution
```

This is the diffusion analogue of using fixed latent vectors in the GAN notebooks.

In [ ]:
fixed_real = real_batch[:8].to(DEVICE)
fixed_noise = torch.randn_like(fixed_real)
fixed_timesteps = torch.tensor(
    [25, 75, 150, 225, T - 1, 50, 125, 250],
    device=DEVICE,
    dtype=torch.long,
)

# Fixed starting noise for generated-sample comparisons later.
generator = torch.Generator(device=DEVICE)
generator.manual_seed(SEED + 1)
fixed_sample_noise = torch.randn(
    16,
    CHANNELS,
    IMAGE_SIZE,
    IMAGE_SIZE,
    generator=generator,
    device=DEVICE,
)

print("fixed_real        :", tuple(fixed_real.shape))
print("fixed_noise       :", tuple(fixed_noise.shape))
print("fixed_timesteps   :", tuple(fixed_timesteps.shape))
print("fixed_sample_noise:", tuple(fixed_sample_noise.shape))

In [ ]:
def fixed_predicted_x0(model, clean_images, noise, timesteps):
    model.eval()
    with torch.no_grad():
        x_t = q_sample(clean_images, timesteps, noise=noise)
        eps_hat = model(x_t, timesteps)
        x0_hat = predict_x0_from_eps(x_t, timesteps, eps_hat)
    return x_t, eps_hat, x0_hat


x_t_fixed_before, eps_fixed_before, x0_fixed_before = fixed_predicted_x0(
    model,
    fixed_real,
    fixed_noise,
    fixed_timesteps,
)

show_images(
    x0_fixed_before[:8],
    title="Fixed Diagnostic: Predicted x₀ Before Full Training",
    rows=2,
    cols=4,
    figsize=(10, 5),
)

## 22 — Full Training Loop

The core training loop stays deliberately explicit. There is no trainer framework hiding the diffusion operations.

Every batch performs:

```text
x₀ → random t → random ε → q_sample → U-Net → MSE → AdamW
```

We also keep a small set of checkpoint **visual outputs**, rather than storing every model state on disk.

In [ ]:
checkpoint_epochs = [0, 1, 2, 5, 10, 15, 20]
checkpoint_images = {}
checkpoint_states = {}
epoch_losses = []

# Store the initial model state on CPU so epoch 0 really means the untrained model.
checkpoint_states[0] = {
    name: tensor.detach().cpu().clone()
    for name, tensor in model.state_dict().items()
}

# Epoch 0 means the randomly initialized model before training.
_, _, checkpoint_images[0] = fixed_predicted_x0(
    model,
    fixed_real,
    fixed_noise,
    fixed_timesteps,
)
checkpoint_images[0] = checkpoint_images[0].detach().cpu()

# Keep the starting noise fixed and sample generated images only at selected checkpoints.
if RUN_TRAINING:
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0

        for x0_batch, _ in loader:
            x0_batch = x0_batch.to(DEVICE, non_blocking=True)
            timesteps = torch.randint(0, T, (x0_batch.shape[0],), device=DEVICE, dtype=torch.long)
            noise = torch.randn_like(x0_batch)
            x_t = q_sample(x0_batch, timesteps, noise=noise)

            predicted_noise = model(x_t, timesteps)
            loss = F.mse_loss(predicted_noise, noise)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

            running_loss += float(loss.detach().cpu())

        epoch_loss = running_loss / max(len(loader), 1)
        epoch_losses.append(epoch_loss)
        print(f"Epoch {epoch:02d}/{EPOCHS} | loss={epoch_loss:.6f}")

        if epoch in checkpoint_epochs:
            checkpoint_states[epoch] = {
                name: tensor.detach().cpu().clone()
                for name, tensor in model.state_dict().items()
            }
            _, _, pred_x0 = fixed_predicted_x0(
                model,
                fixed_real,
                fixed_noise,
                fixed_timesteps,
            )
            checkpoint_images[epoch] = pred_x0.detach().cpu()
else:
    print("RUN_TRAINING=False → skipping the 20-epoch optimization loop.")
    print("Later cells still run using the initialized model, so visualizations remain defined.")

# If EPOCHS was changed, make the final epoch visible in the checkpoint dictionary when possible.
if RUN_TRAINING and EPOCHS not in checkpoint_images:
    checkpoint_states[EPOCHS] = {
        name: tensor.detach().cpu().clone()
        for name, tensor in model.state_dict().items()
    }
    _, _, pred_x0 = fixed_predicted_x0(model, fixed_real, fixed_noise, fixed_timesteps)
    checkpoint_images[EPOCHS] = pred_x0.detach().cpu()

## 23 — Training Loss

A lower MSE generally means the network is getting better at matching the sampled training noise. It does **not** directly imply higher visual sample quality, so we evaluate the generated images separately.

In [ ]:
if epoch_losses:
    epochs_axis = np.arange(1, len(epoch_losses) + 1)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(epochs_axis, epoch_losses, marker="o")
    ax.set_title("Training Loss: Noise Prediction MSE")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE")
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()
else:
    print("No training loss was collected because RUN_TRAINING=False.")

## 24 — Training Progress: Predicted x₀

Every row uses the same diagnostic inputs. As training progresses, the network should improve its ability to infer clean structure from the same noisy observations.

In [ ]:
checkpoint_order = [epoch for epoch in checkpoint_epochs if epoch in checkpoint_images]

fig, axes = plt.subplots(len(checkpoint_order), 4, figsize=(12, 3 * len(checkpoint_order)))
axes = np.atleast_2d(axes)

for row, epoch in enumerate(checkpoint_order):
    images = checkpoint_images[epoch]
    chosen = [0, 1, 2, 3]
    for col, idx in enumerate(chosen):
        image = _to_display_tensor(images[idx])[0].permute(1, 2, 0).numpy()
        axes[row, col].imshow(image)
        axes[row, col].axis("off")
        if row == 0:
            axes[row, col].set_title(f"t={int(fixed_timesteps[idx])}")
    axes[row, 0].set_ylabel(
        f"epoch {epoch}",
        rotation=0,
        labelpad=34,
        va="center",
    )

fig.suptitle("Predicted x₀ Across Training Checkpoints", fontsize=15)
plt.tight_layout()
plt.show()

## 25 — Before Sampling: Reverse-Process Mathematics

The forward posterior has a closed form:

$$
q(x_{t-1}\mid x_t, x_0)=\mathcal N(\tilde\mu_t,\tilde\beta_t I)
$$

where

$$
\tilde\beta_t = \frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t}\beta_t
$$

and

$$
\tilde\mu_t =
\frac{\sqrt{\bar\alpha_{t-1}}\beta_t}{1-\bar\alpha_t}x_0
+
\frac{\sqrt{\alpha_t}(1-\bar\alpha_{t-1})}{1-\bar\alpha_t}x_t.
$$

During generation the true `x₀` is unknown. We replace it with the model estimate `x̂₀` recovered from the predicted noise.

## 26 — Implementing the Reverse Posterior

We already precomputed the two mean coefficients and the posterior variance. The function below assembles them into a batch-shaped mean/variance pair.

In [ ]:
def q_posterior_mean_variance(x0_pred, x_t, t):
    """Compute q(x_{t-1} | x_t, x0_pred) mean and variance."""
    if x0_pred.shape != x_t.shape:
        raise ValueError("x0_pred and x_t must have the same shape")

    coef1 = extract(posterior_mean_coef1, t, x_t.shape)
    coef2 = extract(posterior_mean_coef2, t, x_t.shape)
    variance = extract(posterior_variance, t, x_t.shape)

    posterior_mean = coef1 * x0_pred + coef2 * x_t
    return posterior_mean, variance


x0_test = torch.randn(4, CHANNELS, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
xt_test = torch.randn_like(x0_test)
t_test = torch.randint(0, T, (4,), device=DEVICE, dtype=torch.long)
mean_test, var_test = q_posterior_mean_variance(x0_test, xt_test, t_test)

print("posterior mean shape:", tuple(mean_test.shape))
print("posterior var shape :", tuple(var_test.shape))
assert mean_test.shape == x0_test.shape
assert var_test.shape == (4, 1, 1, 1)

In [ ]:
# Optimizer used by the full multi-epoch training loop.
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
print(f"Optimizer: AdamW | learning rate={LEARNING_RATE}")

## 27 — One Reverse Sampling Step

The reverse step is:

```text
t > 0:
    x_{t-1} = posterior_mean + stochastic noise

t = 0:
    x_0 = posterior_mean
```

The final step must not add another random perturbation. That detail is easy to miss and is worth checking explicitly.

In [ ]:
@torch.no_grad()
def p_sample(model, x_t, t):
    """Perform one DDPM reverse step from x_t to x_{t-1}."""
    was_training = model.training
    model.eval()

    predicted_noise = model(x_t, t)
    x0_pred = predict_x0_from_eps(x_t, t, predicted_noise)
    posterior_mean, posterior_variance_t = q_posterior_mean_variance(x0_pred, x_t, t)

    noise = torch.randn_like(x_t)
    nonzero_mask = (t > 0).to(x_t.dtype).reshape(-1, 1, 1, 1)
    x_prev = posterior_mean + nonzero_mask * torch.sqrt(posterior_variance_t.clamp_min(0.0)) * noise

    if was_training:
        model.train()

    return x_prev


x_test = torch.randn(4, CHANNELS, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
t_test = torch.full((4,), T - 1, device=DEVICE, dtype=torch.long)
with torch.no_grad():
    x_prev_test = p_sample(model, x_test, t_test)
print("Reverse-step output shape:", tuple(x_prev_test.shape))
assert x_prev_test.shape == x_test.shape

## 28 — Full Reverse Sampler

Sampling starts from pure Gaussian noise:

```text
x_T ~ N(0, I)
    ↓
predict noise
    ↓
estimate x₀
    ↓
reverse posterior
    ↓
sample x_{T-1}
    ↓
repeat
    ↓
...
    ↓
x₀
```

For visualization we keep only selected states, not all 300 steps for every sample. That keeps memory use under control.

In [ ]:
@torch.no_grad()
def sample(model, n_samples, initial_noise=None, trajectory_steps=None):
    """Generate images with standard stochastic DDPM sampling."""
    was_training = model.training
    model.eval()

    if initial_noise is None:
        x = torch.randn(
            n_samples,
            CHANNELS,
            IMAGE_SIZE,
            IMAGE_SIZE,
            device=DEVICE,
        )
    else:
        if initial_noise.shape != (n_samples, CHANNELS, IMAGE_SIZE, IMAGE_SIZE):
            raise ValueError(
                "initial_noise must have shape "
                f"({n_samples}, {CHANNELS}, {IMAGE_SIZE}, {IMAGE_SIZE})"
            )
        x = initial_noise.to(DEVICE)

    if trajectory_steps is None:
        trajectory_steps = {T - 1, 250, 200, 150, 100, 50, 0}

    requested = sorted(set(int(v) for v in trajectory_steps if 0 <= int(v) < T), reverse=True)
    trajectory = {}

    for step in range(T - 1, -1, -1):
        if step in trajectory_steps:
            trajectory[step] = x.detach().cpu()
        t_batch = torch.full((n_samples,), step, device=DEVICE, dtype=torch.long)
        x = p_sample(model, x, t_batch)

    trajectory[0] = x.detach().cpu()

    if was_training:
        model.train()

    return x.detach(), trajectory

## 29 — Reverse Diffusion: Watch Noise Become Structure

This is the central visual experiment of the notebook.

We start from one fixed Gaussian noise image and record a few reverse states:

```text
pure noise → vague structure → coarse object → refined output
```

In [ ]:
reverse_demo_noise = fixed_sample_noise[:1].clone()
reverse_steps = {T - 1, 275, 250, 200, 150, 100, 50, 0}

generated_demo, reverse_trajectory = sample(
    model,
    n_samples=1,
    initial_noise=reverse_demo_noise,
    trajectory_steps=reverse_steps,
)

ordered_steps = sorted(reverse_trajectory.keys(), reverse=True)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, step in zip(axes.ravel(), ordered_steps):
    image = _to_display_tensor(reverse_trajectory[step])[0].permute(1, 2, 0).numpy()
    ax.imshow(image)
    ax.set_title(f"t = {step}")
    ax.axis("off")
fig.suptitle("Reverse Diffusion Trajectory", fontsize=15)
plt.tight_layout()
plt.show()

## 30 — Reverse Trajectories From Four Starting Noises

Different initial Gaussian noise tensors should generally lead to different reverse paths. The model is shared; the starting state changes.

In [ ]:
trajectory_noise = fixed_sample_noise[:4].clone()
_, multi_trajectory = sample(
    model,
    n_samples=4,
    initial_noise=trajectory_noise,
    trajectory_steps={T - 1, 250, 200, 150, 100, 50, 0},
)

trajectory_cols = sorted(multi_trajectory.keys(), reverse=True)
fig, axes = plt.subplots(4, len(trajectory_cols), figsize=(16, 10))
for row in range(4):
    for col, step in enumerate(trajectory_cols):
        image = _to_display_tensor(multi_trajectory[step][row])[0].permute(1, 2, 0).numpy()
        axes[row, col].imshow(image)
        axes[row, col].axis("off")
        if row == 0:
            axes[row, col].set_title(f"t={step}")
    axes[row, 0].set_ylabel(f"sample {row + 1}", rotation=0, labelpad=28, va="center")

fig.suptitle("Reverse Trajectories for Multiple Starting Noises", fontsize=15)
plt.tight_layout()
plt.show()

## 31 — Same Starting Noise, Different Checkpoints

We now use exactly the same `x_T` at several training checkpoints. This isolates the effect of the learned model instead of mixing model evolution with a new random starting point.

In [ ]:
checkpoint_sample_epochs = [1, 5, 10, 15, 20]
checkpoint_samples = {}

if RUN_TRAINING:
    original_state = {
        name: tensor.detach().cpu().clone()
        for name, tensor in model.state_dict().items()
    }
    available_epochs = [epoch for epoch in checkpoint_sample_epochs if epoch in checkpoint_states]

    for epoch in available_epochs:
        model.load_state_dict(checkpoint_states[epoch], strict=True)
        samples_at_epoch, _ = sample(
            model,
            n_samples=fixed_sample_noise.shape[0],
            initial_noise=fixed_sample_noise,
            trajectory_steps={T - 1, 150, 0},
        )
        checkpoint_samples[epoch] = samples_at_epoch.detach().cpu()

    # Restore the final trained model before continuing.
    model.load_state_dict(original_state, strict=True)
    print(f"Stored checkpoint states in CPU memory: {available_epochs}")
else:
    samples_at_epoch, _ = sample(
        model,
        n_samples=fixed_sample_noise.shape[0],
        initial_noise=fixed_sample_noise,
        trajectory_steps={T - 1, 150, 0},
    )
    checkpoint_samples["untrained"] = samples_at_epoch.detach().cpu()
    print("RUN_TRAINING=False: fixed-noise samples shown for the initialized model.")

fig, axes = plt.subplots(
    1,
    max(1, len(checkpoint_samples)),
    figsize=(3 * max(1, len(checkpoint_samples)), 3),
)
axes = np.atleast_1d(axes)
for ax, (epoch, images) in zip(axes, checkpoint_samples.items()):
    ax.imshow(_to_display_tensor(images)[0].permute(1, 2, 0).numpy())
    ax.set_title(f"epoch {epoch}")
    ax.axis("off")
fig.suptitle("Fixed Starting Noise Across Model Checkpoints", fontsize=15)
plt.tight_layout()
plt.show()

show_images(
    next(iter(checkpoint_samples.values()))[:16],
    title="Same x_T → Checkpoint Samples",
    rows=4,
    cols=4,
    figsize=(8, 8),
)

## 32 — Final Generated Samples

A sample grid is where we stop looking at individual tensors and ask a broader empirical question: what kind of structure does the trained model produce from noise?

Inspect diversity, rough structure, color coherence, repetition, and artifacts rather than assuming every output should be recognizable.

In [ ]:
final_samples, final_trajectory = sample(
    model,
    n_samples=64,
    initial_noise=None,
    trajectory_steps={T - 1, 250, 200, 150, 100, 50, 0},
)

show_images(
    final_samples,
    title="Generated STL-10 Samples",
    rows=8,
    cols=8,
    figsize=(12, 12),
)

## 33 — Real vs Generated

The comparison below is deliberately simple. Look for:

- diversity across samples
- coherent RGB channels
- object-like structure
- repeated patterns or collapse
- high-frequency artifacts

This is a visual inspection, not a formal generative-quality benchmark.

In [ ]:
real_compare = real_batch[:16].to(DEVICE).detach().cpu()
generated_compare = final_samples[:16].detach().cpu()

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
for col in range(8):
    axes[0, col].imshow(_to_display_tensor(real_compare[col])[0].permute(1, 2, 0).numpy())
    axes[0, col].axis("off")
    axes[0, col].set_title("Real" if col == 0 else "")

    axes[1, col].imshow(_to_display_tensor(generated_compare[col])[0].permute(1, 2, 0).numpy())
    axes[1, col].axis("off")
    axes[1, col].set_title("Generated" if col == 0 else "")

for col in range(8):
    axes[2, col].imshow(_to_display_tensor(real_compare[col + 8])[0].permute(1, 2, 0).numpy())
    axes[2, col].axis("off")

    axes[3, col].imshow(_to_display_tensor(generated_compare[col + 8])[0].permute(1, 2, 0).numpy())
    axes[3, col].axis("off")

fig.suptitle("Real STL-10 vs Generated Samples", fontsize=15)
plt.tight_layout()
plt.show()

## 34 — Denoising Strength by Timestep

For one fixed real image and one fixed noise tensor, we inspect the model at increasing noise levels.

For each timestep we show:

```text
x_t | predicted ε | predicted x₀
```

The question is direct:

> **Can the model infer clean structure even when the input is heavily corrupted?**

In [ ]:
analysis_timesteps = [25, 75, 150, 225, 299]
analysis_x0 = real_batch[0:1].to(DEVICE)
analysis_noise = torch.randn_like(analysis_x0)

fig, axes = plt.subplots(len(analysis_timesteps), 3, figsize=(10, 3 * len(analysis_timesteps)))
for row, step in enumerate(analysis_timesteps):
    t_batch = torch.tensor([step], device=DEVICE, dtype=torch.long)
    x_t = q_sample(analysis_x0, t_batch, noise=analysis_noise)
    with torch.no_grad():
        eps_hat = model(x_t, t_batch)
        x0_hat = predict_x0_from_eps(x_t, t_batch, eps_hat)

    xt_image = _to_display_tensor(x_t)[0].permute(1, 2, 0).numpy()
    eps_display = _normalize_map_for_display(eps_hat[0, 0])
    x0_image = _to_display_tensor(x0_hat)[0].permute(1, 2, 0).numpy()

    axes[row, 0].imshow(xt_image)
    axes[row, 0].axis("off")
    axes[row, 0].set_ylabel(f"t={step}", rotation=0, labelpad=30, va="center")

    axes[row, 1].imshow(eps_display, cmap="magma")
    axes[row, 1].axis("off")

    axes[row, 2].imshow(x0_image)
    axes[row, 2].axis("off")

for title, col in [("x_t", 0), ("predicted ε", 1), ("predicted x̂₀", 2)]:
    axes[0, col].set_title(title)

fig.suptitle("Denoising Strength by Timestep", fontsize=15)
plt.tight_layout()
plt.show()

## 35 — Noise-Prediction Error vs Timestep

We estimate the MSE between the true sampled noise and the model prediction across timestep ranges.

The shape of this curve is an empirical property of the trained model; there is no requirement that every DDPM have exactly the same profile.

In [ ]:
model.eval()
num_eval = 512
steps_for_eval = torch.randint(0, T, (num_eval,), device=DEVICE, dtype=torch.long)
images_for_eval = fixed_real.repeat((math.ceil(num_eval / fixed_real.shape[0]), 1, 1, 1))[:num_eval]
noise_for_eval = torch.randn_like(images_for_eval)

with torch.no_grad():
    x_eval = q_sample(images_for_eval, steps_for_eval, noise=noise_for_eval)
    eps_eval = model(x_eval, steps_for_eval)
    per_item_mse = ((eps_eval - noise_for_eval) ** 2).flatten(1).mean(dim=1)

range_edges = list(range(0, T, 50)) + [T]
range_labels = []
range_means = []
for start, end in zip(range_edges[:-1], range_edges[1:]):
    mask = (steps_for_eval >= start) & (steps_for_eval < end)
    if mask.any():
        range_labels.append(f"{start}–{end - 1}")
        range_means.append(float(per_item_mse[mask].mean().cpu()))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range_labels, range_means, marker="o")
ax.set_title("Noise Prediction Error vs Timestep")
ax.set_xlabel("Timestep range")
ax.set_ylabel("Mean squared error")
ax.grid(alpha=0.2)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 36 — Per-Timestep Predicted x₀

This is a second view of the same denoising experiment. The input becomes progressively less informative, while the model tries to recover a stable estimate of the underlying clean image.

In [ ]:
per_timestep = [25, 75, 150, 225, T - 1]

fig, axes = plt.subplots(1, len(per_timestep) + 1, figsize=(15, 3))
clean_display = _to_display_tensor(analysis_x0)[0].permute(1, 2, 0).numpy()
axes[0].imshow(clean_display)
axes[0].set_title("real x₀")
axes[0].axis("off")

for ax, step in zip(axes[1:], per_timestep):
    t_batch = torch.tensor([step], device=DEVICE, dtype=torch.long)
    x_t = q_sample(analysis_x0, t_batch, noise=analysis_noise)
    with torch.no_grad():
        eps_hat = model(x_t, t_batch)
        x0_hat = predict_x0_from_eps(x_t, t_batch, eps_hat)
    ax.imshow(_to_display_tensor(x0_hat)[0].permute(1, 2, 0).numpy())
    ax.set_title(f"t={step}")
    ax.axis("off")

fig.suptitle("Predicted x₀ at Different Noise Levels", fontsize=15)
plt.tight_layout()
plt.show()

## 37 — Sampling Stochasticity

Standard DDPM sampling injects random noise at intermediate reverse steps.

There are two useful controlled experiments:

1. Same initial `x_T`, with the same random seed/state → reproducible path.
2. Different initial `x_T` → different stochastic paths and usually different outputs.

The exact output also depends on the sequence of random numbers used during the reverse process.

In [ ]:
def sample_with_seed(seed, initial_noise):
    state_cpu = torch.random.get_rng_state()
    state_cuda = torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    result, _ = sample(
        model,
        n_samples=initial_noise.shape[0],
        initial_noise=initial_noise.clone(),
        trajectory_steps={T - 1, 150, 0},
    )
    torch.random.set_rng_state(state_cpu)
    if state_cuda is not None:
        torch.cuda.set_rng_state_all(state_cuda)
    return result


same_a = sample_with_seed(SEED + 10, fixed_sample_noise[:4])
same_b = sample_with_seed(SEED + 10, fixed_sample_noise[:4])
print("Same x_T + same random seed → exact equality:", torch.equal(same_a, same_b))

other_noise = torch.randn_like(fixed_sample_noise[:4])
different = sample_with_seed(SEED + 10, other_noise)
print("Different x_T → mean absolute output difference:", float((same_a - different).abs().mean().cpu()))

## 38 — Lightweight Architecture Ablation

A long second training run would distract from the main lesson, so this experiment only compares parameter counts for a smaller U-Net configuration.

The smaller model is useful for understanding the capacity/runtime trade-off without introducing another training framework.

In [ ]:
small_model = UNet(
    in_channels=CHANNELS,
    base_channels=32,
    time_dim=128,
).to(DEVICE)

print(f"Small U-Net parameters  : {count_parameters(small_model):,}")
print(f"Default U-Net parameters: {count_parameters(model):,}")

small_x = torch.randn(1, CHANNELS, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
small_t = torch.tensor([100], device=DEVICE, dtype=torch.long)
with torch.no_grad():
    small_y = small_model(small_x, small_t)
print("Small model shape check:", tuple(small_y.shape))
assert small_y.shape == small_x.shape

del small_model, small_x, small_t, small_y
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

## 39 — DDPM vs GAN

| Property | GAN | DDPM |
|---|---|---|
| Input | latent vector | Gaussian noise image |
| Main learning signal | adversarial | noise prediction |
| Generation | usually one network pass | many reverse steps |
| Main network | Generator | denoising U-Net |
| Discriminator | yes | no |
| Typical challenge | training can be unstable | sampling is computationally expensive |

These are broad architectural contrasts, not absolute statements about every implementation.

## 40 — DDPM vs VAE

| Property | VAE | DDPM |
|---|---|---|
| Latent structure | explicit latent vector | diffusion trajectory |
| Training | reconstruction + KL | noise prediction |
| Generation | one decoder pass | iterative denoising |
| Main network | encoder + decoder | denoising U-Net |

The key difference is where the generation process lives: a VAE uses an explicit latent code, while DDPM generation is distributed across many denoising steps.

## 41 — Why DDPM Sampling Is Slow

A typical GAN sample looks like:

```text
z → G(z)
```

A DDPM sample looks more like:

```text
x_T
 ↓
x_{T-1}
 ↓
...
 ↓
x_0
```

So one image can require hundreds of U-Net evaluations in the standard formulation used here.

That is one reason later diffusion methods explore alternative samplers. A future DDIM notebook can focus on reducing the number of reverse steps.

## 42 — Failure Modes to Watch For

### 1. Poor denoising
Predicted `x̂₀` remains noisy even after training.

### 2. Color artifacts
RGB channels become incoherent or unnatural.

### 3. Repetitive samples
Different starting noises produce very similar outputs.

### 4. Weak reverse trajectory
The samples do not develop increasingly structured content during denoising.

### 5. Over-smoothing
The model learns rough shapes but little fine detail.

### 6. Training mismatch
Incorrect normalization, timestep handling, broadcasting, or posterior coefficients can break the whole reverse process.

Use the figures above as diagnostics rather than relying on one scalar loss.

## 43 — Common Implementation Mistakes

1. **Mixing `α_t` and `ᾱ_t`** — `α_t` is local; `ᾱ_t` is cumulative.
2. **Forgetting square roots** — the closed-form forward process uses `√ᾱ_t` and `√(1-ᾱ_t)`.
3. **Wrong timestep shape** — use `[B]` before embedding or extraction.
4. **Wrong broadcasting** — schedule values should become `[B, 1, 1, 1]` for image tensors.
5. **Forgetting timestep conditioning** — the U-Net must know the current noise level.
6. **Adding final-step noise** — at `t=0`, return the posterior mean without another stochastic perturbation.
7. **Incorrect posterior coefficients** — small indexing mistakes here can destroy sampling.
8. **Mixing normalized and unnormalized images** — training uses `[-1,1]`; plots use `[0,1]`.
9. **Sampling while the model is still in training mode** — use `model.eval()` through the sampling path.
10. **Storing every reverse state** — for large batches this can consume unnecessary memory; keep selected diagnostic steps.

## 44 — Performance and Memory Notes

This notebook is intentionally configured around:

```text
T = 300
64×64 RGB
compact U-Net
batch size 32
20 epochs
```

The code also uses `torch.no_grad()` for sampling and stores only selected reverse states. The emphasis is **understanding + visualization**, not maximum sample quality.

## 45 — Final Visual Summary Dashboard

The entire notebook can be summarized by six views:

```text
forward corruption
reverse denoising
predicted x₀
training loss
noise-prediction error
final generated samples
```

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. Forward trajectory
trajectory_pick = [0, 50, 100, 200, T - 1]
for step in trajectory_pick:
    t_one = torch.tensor([step], device=DEVICE, dtype=torch.long)
    noisy = q_sample(source_image, t_one, noise=fixed_forward_noise)
    image = _to_display_tensor(noisy)[0].permute(1, 2, 0).numpy()
    # First panel: last selected forward state is shown as the representative corruption.
    if step == T - 1:
        axes[0, 0].imshow(image)
        axes[0, 0].set_title("Forward: x_t at max noise")
axes[0, 0].axis("off")

# 2. Reverse terminal state
summary_reverse = final_trajectory[0]
axes[0, 1].imshow(_to_display_tensor(summary_reverse)[0].permute(1, 2, 0).numpy())
axes[0, 1].set_title("Reverse: x₀")
axes[0, 1].axis("off")

# 3. Predicted x0 diagnostic
axes[0, 2].imshow(_to_display_tensor(x0_fixed_before[0])[0].permute(1, 2, 0).numpy())
axes[0, 2].set_title("Predicted x₀ diagnostic")
axes[0, 2].axis("off")

# 4. Loss
if epoch_losses:
    axes[1, 0].plot(np.arange(1, len(epoch_losses) + 1), epoch_losses)
    axes[1, 0].set_title("Training loss")
    axes[1, 0].set_xlabel("Epoch")
    axes[1, 0].set_ylabel("MSE")
    axes[1, 0].grid(alpha=0.2)
else:
    axes[1, 0].text(0.5, 0.5, "RUN_TRAINING=False", ha="center", va="center")
    axes[1, 0].set_title("Training loss")
    axes[1, 0].axis("off")

# 5. Error curve
axes[1, 1].plot(range(len(range_means)), range_means, marker="o")
axes[1, 1].set_title("Noise prediction error")
axes[1, 1].set_xlabel("Timestep range bin")
axes[1, 1].set_ylabel("MSE")
axes[1, 1].grid(alpha=0.2)

# 6. Final generated montage
for idx in range(4):
    image = _to_display_tensor(final_samples[idx])[0].permute(1, 2, 0).numpy()
    # Use the panel itself as a 2x2 mini-montage by averaging four examples.
images = _to_display_tensor(final_samples[:4]).numpy().transpose(0, 2, 3, 1)
canvas = np.zeros((IMAGE_SIZE * 2, IMAGE_SIZE * 2, 3), dtype=np.float32)
canvas[:IMAGE_SIZE, :IMAGE_SIZE] = images[0]
canvas[:IMAGE_SIZE, IMAGE_SIZE:] = images[1]
canvas[IMAGE_SIZE:, :IMAGE_SIZE] = images[2]
canvas[IMAGE_SIZE:, IMAGE_SIZE:] = images[3]
axes[1, 2].imshow(canvas)
axes[1, 2].set_title("Final generated samples")
axes[1, 2].axis("off")

fig.suptitle("DDPM From Scratch — Visual Summary Dashboard", fontsize=17)
plt.tight_layout()
plt.show()

## 46 — Final Mental Model

### Training

```text
real image x₀
     ↓
sample timestep t
     ↓
sample Gaussian noise ε
     ↓
construct x_t
     ↓
U-Net(x_t, t)
     ↓
predict ε̂
     ↓
MSE(ε, ε̂)
     ↓
update U-Net
```

### Sampling

```text
x_T ~ N(0, I)
     ↓
predict noise
     ↓
estimate x₀
     ↓
compute reverse posterior
     ↓
sample x_{T-1}
     ↓
repeat
     ↓
...
     ↓
x₀
```

> **DDPM = learned iterative denoising**

## 47 — Mathematical Cheat Sheet

$$
\alpha_t = 1-\beta_t
$$

$$
\bar\alpha_t = \prod_{s=1}^{t}\alpha_s
$$

$$
x_t = \sqrt{\bar\alpha_t}x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon
$$

$$
\epsilon_\theta(x_t,t)
$$

$$
\hat{x}_0 = \frac{x_t-\sqrt{1-\bar\alpha_t}\,\epsilon_\theta(x_t,t)}{\sqrt{\bar\alpha_t}}
$$

$$
L_{simple}=\mathbb E\left[\|\epsilon-\epsilon_\theta(x_t,t)\|^2\right]
$$

$$
q(x_{t-1}|x_t,x_0)=\mathcal N(\tilde\mu_t,\tilde\beta_t I)
$$

## 48 — What Was Implemented

| Component | Implemented |
|---|---|
| STL-10 preprocessing | Yes |
| Beta schedule | Yes |
| Forward diffusion | Yes |
| `q_sample` | Yes |
| Time embedding | Yes |
| U-Net | Yes |
| Noise prediction | Yes |
| DDPM loss | Yes |
| Training loop | Yes |
| Predicted x₀ | Yes |
| Reverse posterior | Yes |
| Reverse sampler | Yes |
| Visualization | Yes |
| Generated samples | Yes |

## 49 — Diffusion Roadmap

```text
06 — Diffusion Fundamentals
    Mathematics + intuition

07 — DDPM From Scratch
    Full practical implementation

08 — DDIM
    Faster / alternative sampling

09 — Conditional Diffusion
    Conditioning generation

10 — Classifier-Free Guidance
    Stronger conditional control

11 — Latent Diffusion
    Diffusion in latent space
```

The later notebooks build on this DDPM core rather than introducing a completely different mental model.

## 50 — Try It Yourself

1. Change `T` from 300 to 100 and inspect the schedule.
2. Replace the linear beta schedule with another simple schedule.
3. Increase or decrease the U-Net channel width.
4. Change `TIME_EMBED_DIM` and compare parameter counts.
5. Compare predicted `x̂₀` at different timesteps.
6. Train for more epochs and inspect whether visual quality changes with loss.
7. Generate more samples and inspect diversity.
8. Compare two different fixed starting noise tensors.
9. Visualize the effect of changing `BETA_END`.
10. Compare different timestep bins for noise-prediction error.
11. Remove one skip connection and observe the effect on shape/quality.
12. Modify the residual block and compare the training behavior.

Do not optimize all experiments at once. Change one variable, keep the rest fixed, and make the visual comparison meaningful.

## 51 — Optional Advanced Notes

### Score matching

Diffusion models are closely connected to score-based generative modeling. The connection becomes useful when studying the broader family of score-based and SDE formulations.

### DDIM

DDIM provides an alternative reverse formulation that can reduce the number of sampling steps. It is the natural next notebook for studying faster generation.

### Classifier-Free Guidance

Conditional diffusion can use conditional and unconditional predictions to steer generation more strongly. That belongs in a later notebook so the unconditional DDPM core stays clear here.

## 52 — Implementation Checklist

Before moving on, verify these points:

- [ ] `t` uses the internal convention `0 ... T-1`.
- [ ] `extract` returns `[B,1,1,1]`.
- [ ] `q_sample` preserves image shape.
- [ ] The U-Net output has exactly the same shape as its input image.
- [ ] Time embeddings are injected into residual blocks.
- [ ] Training predicts noise rather than pixels.
- [ ] Predicted `x̂₀` uses the same schedule convention as `q_sample`.
- [ ] Posterior coefficients are precomputed once.
- [ ] Sampling runs from `T-1` down to `0`.
- [ ] No extra random noise is added at `t=0`.
- [ ] Sampling uses `torch.no_grad()` and `model.eval()`.
- [ ] Visualization converts model tensors from `[-1,1]` to `[0,1]`.

## 53 — Closing

The essential loop is now concrete:

```text
q(x_t | x₀)
      ↓
noise schedule
      ↓
U-Net
      ↓
ε_θ(x_t,t)
      ↓
MSE noise prediction
      ↓
predicted x₀
      ↓
reverse posterior
      ↓
iterative sampling
      ↓
generated STL-10 images
```

You can now move from the mathematics of diffusion to alternative samplers and conditional generation without treating the reverse process as a black box.